# 01. Business Understanding & Problem Framing


## Latar Belakang

Pemerintah pusat maupun daerah memiliki kewajiban untuk merumuskan kebijakan tata ruang yang presisi, seperti Rencana Tata Ruang Wilayah (RTRW) dan pengawasan Rencana Detail Tata Ruang (RDTR). Kebijakan ini sangat bergantung pada pemutakhiran data tutupan dan penggunaan lahan (LULC). Sayangnya, instansi pemerintahan sering kali menghadapi kendala keterlambatan data karena audit lapangan secara manual memakan waktu berbulan-bulan dan menghabiskan anggaran negara yang besar. Akibatnya, pelanggaran zonasi—seperti konversi lahan sawah produktif menjadi perumahan atau deforestasi mangrove ilegal—sering terlambat ditindak.

Dari kacamata seorang programmer atau data engineer di sektor gov-tech, ketersediaan citra satelit berkala seperti Sentinel-2A merupakan sumber data mentah (raw data) bervolume raksasa yang tidak bisa lagi diproses secara manual oleh analis manusia. Tantangan utamanya adalah bagaimana membangun sebuah sistem otomasi yang dapat menarik (ingest) data satelit, memproses piksel-piksel tersebut secara komputasional, dan mendistribusikan hasil klasifikasinya kepada para pemangku kebijakan dalam bentuk dashboard atau API yang terintegrasi dengan sistem informasi geografis (GIS) pemerintah.

Tujuan Bisnis

Proyek ini bertujuan untuk menjembatani kebutuhan analitik pemerintah dengan solusi rekayasa perangkat lunak melalui penerapan Machine Learning. Secara spesifik, tujuan ini dibagi menjadi dua dimensi utama:

Dimensi Kebijakan Publik (Dampak Sosio-Ekonomi):

Memberikan landasan data yang mutakhir dan real-time bagi pemerintah untuk mengambil keputusan strategis. Dengan memetakan lahan ke dalam 6 kelas utama (Sawah, Bangunan, Mangrove, Hutan, Laut, dan Danau), pemerintah dapat secara otomatis memantau ketahanan pangan (luas sawah), laju urbanisasi (ekspansi bangunan), pelestarian lingkungan (mangrove dan hutan), serta batas sumber daya air. Hal ini mempercepat audit tata ruang dari hitungan bulan menjadi hitungan jam.

Dimensi Pengembangan Perangkat Lunak (Teknis & Operasional):

Membangun pipeline pemrosesan data geospasial yang skalabel dan efisien. Sebagai programmer, tujuan teknisnya adalah merancang eksperimen untuk membandingkan performa algoritma Naive Bayes dan K-Nearest Neighbors (KNN). Evaluasi tidak hanya berfokus pada akurasi klasifikasi kelas lahan, tetapi juga pada efisiensi komputasi (waktu training dan inference), sehingga model yang terpilih dapat di-deploy ke dalam arsitektur peladen (server) pemerintah tanpa membebani infrastruktur secara berlebihan.

Kriteria Keberhasilan (Success Criteria)

Keberhasilan Teknis: Model Machine Learning yang terpilih (antara Naive Bayes atau KNN) mampu mengklasifikasikan 6 kelas LULC dengan metrik akurasi (misalnya F1-Score) di atas batas ambang yang disepakati, dan script pipeline dapat berjalan secara otomatis tanpa intervensi manual.

Keberhasilan Bisnis/Kebijakan: Terciptanya purwarupa aplikasi atau layanan API yang memampukan analis tata ruang pemerintah untuk melihat perubahan lahan secara interaktif, yang berujung pada efisiensi waktu operasional survei lapangan hingga lebih dari 60%.

In [1]:
from pathlib import Path
from IPython.display import Markdown, display

BASE_DIR = Path.cwd()
INDEX_FILE = BASE_DIR / "index.md"

if not INDEX_FILE.exists():
    # Jika dijalankan dari subfolder atau path lain
    INDEX_FILE = Path("index.md")

if INDEX_FILE.exists():
    with open(INDEX_FILE, "r", encoding="utf-8") as f:
        isi_index = f.read()
    display(Markdown(isi_index))
else:
    print(f"Peringatan: Berkas {INDEX_FILE} tidak ditemukan.")

# LULC

LULC adalah singkatan dari Land Use and Land Cover (Penggunaan Lahan dan Tutupan Lahan). Informasi mengenai tutupan dan penggunaan lahan sangat penting dalam perencanaan wilayah, manajemen sumber daya alam, hingga memantau perubahan iklim. Pendataan lahan secara manual dengan terjun langsung ke lapangan memakan waktu dan biaya operasional yang sangat besar. Untungnya, satelit observasi bumi seperti Sentinel-2A mengorbit dan mengambil citra bumi secara berkala. Namun, membedakan mana Sawah, Danau, atau Hutan dari citra satelit mentah tetap memerlukan pengamatan khusus dari tenaga ahli.


### 2. Rumusan Masalah
1. **Biaya dan Waktu Survei Manual**: Pendataan lahan secara manual dengan survei lapangan (terestrial) di wilayah seluas Provinsi Jawa Timur (~48.000 km²) membutuhkan waktu berbulan-bulan dan anggaran operasional yang sangat besar.
2. **Kompleksitas Citra Satelit Mentah**: Citra satelit Sentinel-2A mentah memiliki multi-saluran panjang gelombang yang tidak dapat diinterpretasi langsung secara cepat oleh pengambil kebijakan tanpa keahlian khusus penginderaan jauh (*remote sensing*).
3. **Kebutuhan Otomatisasi Klasifikasi Spasial**: Diperlukan sebuah sistem pembelajaran mesin (*Machine Learning*) yang mampu mengklasifikasikan piksel citra satelit ke dalam 6 kelas tutupan lahan secara otomatis, cepat, akurat, dan dapat direproduksi (*reproducible*) secara berkala.

### 3. Tujuan Bisnis dan Tujuan Data Science

#### A. Tujuan Bisnis (*Business Objectives*)
- **Penyediaan Peta Tematik**: Menyediakan peta digital tutupan lahan (LULC) Provinsi Jawa Timur yang mutakhir dan akurat.
- **Dukungan Keputusan Terukur**: Mendukung perencanaan tata ruang wilayah (RTRW), inventarisasi lahan pertanian pangan berkelanjutan (LP2B), pemantauan kawasan hutan dan konservasi mangrove, serta deteksi alih fungsi lahan secara periodik.

#### B. Tujuan Data Science (*Data Science Objectives*)
- **Pemodelan Spasial 6 Kelas**: Membangun model klasifikasi mutakhir berbasis algoritma **Support Vector Machine (SVM)** dengan kernel Radial Basis Function (RBF) untuk membedakan 6 kelas tutupan lahan: *Sawah, Bangunan, Hutan, Danau, Laut, dan Mangrove*.
- **Mitigasi Kebocoran Data Spasial**: Mengimplementasikan agregasi centroid per poligon guna mengeliminasi autokorelasi spasial (*spatial autocorrelation leakage*) saat pembagian data latih dan data uji.
- **Diseminasi Interaktif**: Menyajikan hasil klasifikasi ground truth, evaluasi model, dan prediksi raster skala penuh Jawa Timur ke dalam visualisasi peta interaktif web-GIS Folium dan dashboard Streamlit.

### 4. Pemangku Kepentingan dan Pengguna Hasil (*Stakeholders*)
- **Badan Perencanaan Pembangunan Wilayah**: Menggunakan peta LULC sebagai acuan zonasi spasial dan evaluasi pola ruang wilayah.
- **Dinas Lingkungan Hidup & Kehutanan**: Memonitor keutuhan tutupan vegetasi hutan lindung, sebaran mangrove pesisir, dan perubahan badan air alami.
- **Dinas Pertanian & Ketahanan Pangan**: Mengestimasi luasan sawah produktif dan mendeteksi alih fungsi lahan sawah ke pemukiman/bangunan.
- **Peneliti & Mahasiswa Akademik**: Memanfaatkan metodologi openEO, mitigasi spatial leakage, dan klasifikasi raster multispektral sebagai referensi riset data science spasial.

### 5. Kriteria Keberhasilan (*Success Criteria*)
Kriteria keberhasilan ditetapkan sebagai target objektif sebelum pemodelan dilakukan:
- **Target Akurasi & Macro F1 Data Uji**: Model SVM ditargetkan mencapai Akurasi minimum $\ge 85\%$ dan skor Macro F1 $\ge 0.85$ pada data uji poligon yang belum pernah dilihat saat pelatihan (*unseen test data*).
- **Integritas Metodologi Spasial**: Evaluasi bebas dari kebocoran data spasial dengan pembagian data pada tingkat poligon (*polygon-level stratified train-test split*).
- **Fungsionalitas Sistem & Peta**: Terbentuknya peta web interaktif bebas blokir untuk data ground truth, komparasi spasial prediksi SVM, dan prediksi raster skala penuh Provinsi Jawa Timur, serta tersedianya dashboard Streamlit yang stabil tanpa galat.

### 6. Ruang Lingkup dan Asumsi (*Scope & Assumptions*)
- **Wilayah Studi**: Seluruh daratan dan perairan Provinsi Jawa Timur, termasuk gugusan pulau terluar seperti Kepulauan Kangean.
- **Kelas LULC (6 Kelas)**: 1: Sawah, 2: Bangunan, 3: Hutan, 4: Danau, 5: Laut, 6: Mangrove.
- **Sumber Citra**: Sentinel-2A Level-2A (Bottom of Atmosphere / BOA Reflectance) yang diunduh dan diproses melalui platform openEO (Copernicus Data Space Ecosystem / CDSE).
- **Periode Waktu**: Komposit citra musim kemarau (*dry season*, Agustus 2024) untuk meminimalisir tutupan awan persisten dan menjaga konsistensi pantulan spektral.
- **Metode Pembelajaran**: Menggunakan metode tunggal terstandarisasi: **Support Vector Machine (SVM)** dengan RBF kernel ($C=10, \gamma=	ext{'scale'}$) dalam `Pipeline` bersama `StandardScaler`.

### 7. Batasan dan Risiko (*Constraints & Risks*)
- **Tutupan Awan dan Bayangan**: Area tropis sering mengalami tutupan awan kumulus; pemilihan citra dengan *cloud cover* $<30\%$ dan agregasi median temporal diterapkan untuk mitigasi.
- **Efek Piksel Campuran (*Mixed Pixel*)**: Citra raster regional Jawa Timur diunduh pada resolusi spasial ~250 m per piksel untuk efisiensi komputasi, sementara sampel poligon berasal dari piksel 10–20 m. Hal ini berpotensi memicu *domain shift* dan ketidakpastian klasifikasi pada batas-batas objek sempit (misal sempadan sungai atau mangrove tepi pantai).
- **Dominasi Wilayah Laut**: Area laut Jawa Timur sangat luas dibandingkan daratan sehingga inferensi raster penuh memerlukan pengelolaan memori batch agar tidak terjadi kehabisan RAM.
- **Ukuran Sampel Poligon Terbatas**: Terdapat 357 poligon sampel ground truth terverifikasi, sehingga data uji independen berjumlah 72 poligon. Oleh karena itu, *stratified split* wajib dijaga agar tiap kelas terwakili secara proporsional.

### 8. Gambaran Alur Proyek (*End-to-End Pipeline*)

| No | Tahapan Notebook | Deskripsi Utama | Keluaran Utama |
|---|---|---|---|
| **01** | `01_business_understanding.ipynb` | Perumusan masalah bisnis, tujuan data science, dan kriteria keberhasilan | `output/01_business_understanding.md` |
| **02** | `02_data_understanding.ipynb` | Eksplorasi dataset piksel Sentinel-2A, distribusi kelas, dan respons spektral | `output/02_distribusi_kelas_piksel.csv`, `output/02_*.png` |
| **03** | `03_data_preparation.ipynb` | Centroid aggregation poligon (mitigasi spatial leakage) dan stratified 80:20 split | `output/df_poligon_split.csv` |
| **04** | `04_modeling_evaluasi_svm.ipynb` | Pelatihan Pipeline StandardScaler + SVM (RBF) dan evaluasi komprehensif | `output/model_svm.joblib`, `output/df_poligon_pred_svm.csv` |
| **05** | `05_visualisasi_peta_folium.ipynb` | Peta interaktif poligon data aktual (ground truth) per kelas | `output/peta_aktual_lulc.html` |
| **06** | `06_visualisasi_prediksi_model.ipynb` | Peta interaktif evaluasi prediksi SVM (benar vs salah pada data uji) | `output/peta_prediksi_svm.html`, `output/06_*.png` |
| **07** | `07_visualisasi_prediksi_raster_jatim.ipynb` | Inferensi batch SVM pada citra raster Sentinel-2 Jawa Timur (~250 m) | `output/prediksi_lulc_jatim_svm.tif`, `output/peta_prediksi_raster_jatim.html` |
| **08** | `08_streamlit_app.ipynb` | Penulisan modul web dashboard Streamlit interaktif (`app.py` & `requirements.txt`) | `app.py`, `requirements.txt` |
| **09** | `09_kesimpulan.ipynb` | Sintesis temuan ilmiah, capaian metrik, dan rekomendasi operasional | Ringkasan eksekutif proyek |

In [2]:
# Menyimpan ringkasan Business Understanding ke output/01_business_understanding.md
ringkasan_md = '''# Ringkasan Business Understanding: LULC Jawa Timur (Sentinel-2A & SVM)

## 1. Tujuan Bisnis
Menyediakan peta tematik klasifikasi Land Use Land Cover (LULC) Provinsi Jawa Timur 6 kelas yang akurat dan dapat diperbarui secara berkala untuk mendukung perencanaan tata ruang wilayah (RTRW), perlindungan lahan pertanian (LP2B), dan pemantauan kelestarian lingkungan pesisir serta hutan.

## 2. Tujuan Data Science
Membangun model klasifikasi 6 kelas (Sawah, Bangunan, Hutan, Danau, Laut, Mangrove) berbasis Support Vector Machine (SVC kernel RBF, C=10, gamma='scale') dengan fitur 5 band spektral (B02, B03, B04, B08, B11) dan 4 indeks spektral (NDVI, NDWI, MNDWI, NDBI). Evaluasi dilakukan pada tingkat agregasi poligon untuk mencegah kebocoran spasial.

## 3. Kriteria Keberhasilan
- Akurasi pengujian >= 85% dan Macro F1 >= 0.85 pada data uji independen.
- Tidak terjadi spatial data leakage (pembagian data berbasis poligon).
- Sistem visualisasi Web-GIS interaktif dan dashboard Streamlit berfungsi penuh tanpa galat.

## 4. Ruang Lingkup dan Batasan
- Wilayah studi: Provinsi Jawa Timur (termasuk Kepulauan Kangean).
- Citra satelit: Sentinel-2A Level-2A dari openEO CDSE (komposit Agustus 2024).
- Metode tunggal: Pipeline StandardScaler + SVM (SVC RBF).
- Resolusi data latih: 10-20 m (agregasi poligon ground truth); Resolusi raster regional: ~250 m.
'''

OUT_DIR = BASE_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
berkas_keluaran = OUT_DIR / "01_business_understanding.md"

with open(berkas_keluaran, "w", encoding="utf-8") as f:
    f.write(ringkasan_md.strip())

print(f"File tersimpan: {berkas_keluaran.resolve()}")
print(f"Ukuran berkas: {berkas_keluaran.stat().st_size} bytes")

File tersimpan: D:\tugas psd\website-jupyterbook\UTS\output\01_business_understanding.md
Ukuran berkas: 1373 bytes
